# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions

- Step 1: Run the followings cell to get connected to your MongoDB. Make sure your credential is in dotenv file.
- Step 2: Put your answer inside each function. Please do not construct your own function.
- Step 3: You can test your function under test section.
- Step 4. Run test my function to confirm if my code is working.


### Connections

In [109]:
import os
import pymongo
from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

In [110]:
# Load environment variables from .env file
load_dotenv()
MONGODB_URI = os.getenv("MONGODB_URI")
if not MONGODB_URI:
    raise ValueError(
        "❌ MONGODB_URI not found!\n"
        "Please create a .env file with your MongoDB credentials.\n"
        "See README.md for setup instructions."
    )
client = MongoClient(MONGODB_URI, server_api=ServerApi("1"))
# Send a ping to confirm a successful connection
try:
    client.admin.command("ping")
    print("✅ Successfully connected to MongoDB!")
except Exception as e:
    print(e)

✅ Successfully connected to MongoDB!


In [111]:
db = client.sample_mflix
movies = db.movies
print(f"📊 Database: {db.name}")
print(f"📁 Collection: movies ({movies.count_documents({})} documents)")

📊 Database: sample_mflix
📁 Collection: movies (21349 documents)



### Question 1

Question: From the `movies` collection, return the documents with the `plot` that starts with `"war"` in acending order of released date, print only title, plot and released fields. Limit the result to 5.


**Answer:**

In [112]:
# ===============================================================================
# Question 1 - return the plot which starts with "war" in ascending released date
# ===============================================================================

for m in movies.find(
    {"plot": {"$regex": "^War"}},
).sort("released", pymongo.ASCENDING).limit(10):
    print(f"Plot: {m['plot']}")
    print(f"_id: {m['_id']}")
    print(f"Title: {m['title']}")
    print(f"Released: {m['released']}")
    print(f"Rated: {m.get('rated', 'N/A')}")
    print()


Plot: Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet.
_id: 573a1398f29313caabce91ec
Title: Nausicaè of the Valley of the Wind
Released: 1984-03-11 00:00:00
Rated: PG

Plot: Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet.
_id: 573a1398f29313caabce9508
Title: Nausicaè of the Valley of the Wind
Released: 1984-03-11 00:00:00
Rated: PG

Plot: Warlords Kagetora and Takeda each wish to prevent the other from gaining hegemony in feudal Japan. The two samurai leaders pursue one another across the countryside, engaging in massive ...
_id: 573a1398f29313caabcebfc6
Title: Heaven and Earth
Released: 1991-02-08 00:00:00
Rated: PG-13

Plot: Warning! This synopsis contains spoilers Bajo las estrellas (beneath the stars) features the selfish...
_id: 573a13b5f29313caabd44f06
Title: Under the Stars
Released: 2007-06-15 00:00:

### Question 2

Question: Group by `rated` and count the number of movies in each.

**Answer:**

In [113]:
# ================================================================
# Question 2 Group by rated and count the number of movies in each
# ================================================================

pipeline = [
    # 1. Group by the 'rated' field and sum up the documents
    {
        "$group": {
            "_id": "$rated",
            "movie_count": { "$sum": 1 } # Counts 1 for each movie in that rating group
        }
    },
    # 2. Sort the results alphabetically by rating name (or change to 'movie_count' to sort by size)
    {
        "$sort": {
            "_id": pymongo.ASCENDING 
        }
    }
]

# Run the aggregation pipeline
results = movies.aggregate(pipeline)

# Loop through and print the breakdown
for rating_summary in results:
    # Documents without a 'rated' field will show up under an '_id' of None
    rating = rating_summary['_id'] if rating_summary['_id'] is not None else "Unrated / Missing"
    count = rating_summary['movie_count']
    
    print(f"Rating: {rating} | Total Movies: {count}")

Rating: Unrated / Missing | Total Movies: 9894
Rating: AO | Total Movies: 3
Rating: APPROVED | Total Movies: 709
Rating: Approved | Total Movies: 5
Rating: G | Total Movies: 477
Rating: GP | Total Movies: 44
Rating: M | Total Movies: 37
Rating: Not Rated | Total Movies: 1
Rating: OPEN | Total Movies: 1
Rating: PASSED | Total Movies: 181
Rating: PG | Total Movies: 1852
Rating: PG-13 | Total Movies: 2321
Rating: R | Total Movies: 5537
Rating: TV-14 | Total Movies: 89
Rating: TV-G | Total Movies: 59
Rating: TV-MA | Total Movies: 60
Rating: TV-PG | Total Movies: 76
Rating: TV-Y7 | Total Movies: 3


##### To check if correct, the total number of movies from above is 21,349 which is correct. 

In [114]:
# Pass an empty dictionary {} to count all documents in the collection
total_movies = movies.count_documents({})

print(f"Total Movies: {total_movies}")

Total Movies: 21349



### Question 3

Question: Count the number of movies with 3 comments or more.


**Answer:**


In [115]:
# ============================================================================
# Question 3 Count the no. of movies with 3 comments or more
# ============================================================================

pipeline = [
    # 1. Group by movie_id and count how many comments each movie has
    {
        "$group": {
            "_id": "$movie_id",
            "comment_count": { "$sum": 1 }
        }
    },
    # 2. IMMEDIATELY drop any movie with less than 3 comments
    {
        "$match": {
            "comment_count": { "$gte": 3 }
        }
    },
    # 3. ONLY lookup data for the few movies that passed the filter
    {
        "$lookup": {
            "from": "movies",         # Join with the movies collection
            "localField": "_id",      # The movie_id from comments grouping
            "foreignField": "_id",    # The _id inside the movies collection
            "as": "movie_details"
        }
    },
    # 4. Flatten the movie_details array into a single document object
    {
        "$unwind": "$movie_details"
    },
    # 5. Clean up the output structure to show exactly what you want
    {
        "$project": {
            "_id": 1,
            "comment_count": 1,
            "title": "$movie_details.title",
            "plot": "$movie_details.plot",
            "released": "$movie_details.released"
        }
    }
]

# CRUCIAL: Run this on the COMMENTS collection
results = db.comments.aggregate(pipeline)

for movie in results:
    print(f"Title: {movie.get('title', 'N/A')}")
    print(f"Plot: {movie.get('plot', 'N/A')}")
    print(f"Released: {movie.get('released', 'N/A')}")
    print(f"Comments Count: {movie['comment_count']}")
    print("-" * 40)


Title: Quarantine
Plot: A television reporter and her cameraman are trapped inside a building quarantined by the CDC after the outbreak of a mysterious virus which turns humans into bloodthirsty killers.
Released: 2008-10-10 00:00:00
Comments Count: 121
----------------------------------------
Title: The End of a Mystery
Plot: Joaquin comes back to Granada in the eighties trying to find out about something happened when he was a child and the Spanish Civil War was going on. He helped an unknown man who survived ...
Released: 2003-01-31 00:00:00
Comments Count: 122
----------------------------------------
Title: Liar Liar
Plot: A fast track lawyer can't lie for 24 hours due to his son's birthday wish after the lawyer turns his son down for the last time.
Released: 1997-03-21 00:00:00
Comments Count: 128
----------------------------------------
Title: Stand Clear of the Closing Doors
Plot: The story of an autistic youth named Ricky who, after a particularly difficult day at school, escap